# 03 Payment EDA
## Exploratory Data Analysis — olist_order_payments_dataset.csv
### Dataset: Brazilian E-Commerce Public Dataset (Kaggle)

## 1. Imports

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Load Data

In [ ]:
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)

payment_file_path = os.path.join(project_root, 'Data', 'raw', 'Olist_ecom_data', 'olist_order_payments_dataset.csv')
payment = pd.read_csv(payment_file_path)


## 3. Initial Data Check

In [ ]:
payment.info()


In [ ]:
payment.head(10)


In [ ]:
print('Missing Values:')
print(payment.isnull().sum())
print('\nBasic Stats:')
payment.describe()


## 4. Data Quality Checks

In [ ]:
# Orders with 0 payment value
zero_payment_orders = payment[payment['payment_value'] == 0]
print(f'Orders with 0 payment value: {zero_payment_orders.shape[0]}')
print(payment[payment['payment_value'] == 0]['payment_type'].value_counts())


In [ ]:
# Orders with multiple payments (duplicate order_ids)
duplicate_order_ids = payment[payment.duplicated(subset='order_id', keep=False)]
num_duplicate_order_ids = duplicate_order_ids['order_id'].nunique()
print(f'Orders with multiple payments: {num_duplicate_order_ids}')


In [ ]:
# All payment types before cleaning
print(f'Payment types before cleaning:')
print(payment['payment_type'].value_counts())


## 5. Data Cleaning
> **Note:** Cleaning is done BEFORE KPI calculations to ensure accurate results.

In [ ]:
# Drop undefined + 0 value rows (genuine data errors)
payment = payment[~(
    (payment['payment_type'] == 'not_defined') &
    (payment['payment_value'] == 0)
)]

print(f'Rows after cleaning: {len(payment)}')
print(f'Payment types after cleaning:')
print(payment['payment_type'].value_counts())


## 6. Revenue KPIs — Transaction Level

In [ ]:
total_revenue = payment['payment_value'].sum()
aov_transaction = payment['payment_value'].mean()
median_transaction = payment['payment_value'].median()
max_transaction = payment['payment_value'].max()
min_transaction = payment['payment_value'][payment['payment_value'] > 0].min()

print(f'Total Revenue:             R$ {total_revenue:,.2f}')
print(f'Avg Transaction Value:     R$ {aov_transaction:.2f}')
print(f'Median Transaction Value:  R$ {median_transaction:.2f}')
print(f'Max Transaction Value:     R$ {max_transaction:.2f}')
print(f'Min Transaction Value:     R$ {min_transaction:.2f}')


## 7. Revenue KPIs — Order Level
> **Note:** More accurate AOV — accounts for multi-payment orders by grouping by order_id first.

In [ ]:
order_level = payment.groupby('order_id').agg(
    total_value=('payment_value', 'sum'),
    num_payments=('payment_type', 'count'),
    payment_types=('payment_type', lambda x: ','.join(x.unique()))
).reset_index()

aov_order = order_level['total_value'].mean()
median_order = order_level['total_value'].median()

print(f'True AOV (per order):      R$ {aov_order:.2f}')
print(f'Median Order Value:        R$ {median_order:.2f}')
print(f'Total Unique Orders:       {order_level["order_id"].nunique()}')


## 8. Payment Method Analysis

In [ ]:
# Count and % by payment type
payment_count = payment['payment_type'].value_counts().reset_index()
payment_count.columns = ['payment_type', 'count']
payment_count['percentage'] = (payment_count['count'] / len(payment) * 100).round(2)
print(payment_count)


In [ ]:
# Revenue by payment type
revenue_by_type = payment.groupby('payment_type')['payment_value'].sum().reset_index()
revenue_by_type.columns = ['payment_type', 'total_revenue']
revenue_by_type['revenue_pct'] = (revenue_by_type['total_revenue'] / total_revenue * 100).round(2)
revenue_by_type = revenue_by_type.sort_values('total_revenue', ascending=False)
print(revenue_by_type)


In [ ]:
# Average order value by payment type
avg_by_type = payment.groupby('payment_type')['payment_value'].mean().reset_index()
avg_by_type.columns = ['payment_type', 'avg_value']
avg_by_type = avg_by_type.sort_values('avg_value', ascending=False)
print(avg_by_type)


In [ ]:
# High value orders (>1000) by payment type
high_value_orders = payment[payment['payment_value'] > 1000]
total_high_value = high_value_orders.shape[0]
high_value_by_type = high_value_orders['payment_type'].value_counts().reset_index()
high_value_by_type.columns = ['payment_type', 'count']
high_value_by_type['pct_of_high_value'] = (high_value_by_type['count'] / total_high_value * 100).round(2)
print(f'High value orders (>R$1000): {total_high_value} ({total_high_value/len(payment)*100:.2f}% of all orders)')
print(high_value_by_type)


### Payment Method Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.barplot(data=payment_count, x='count', y='payment_type', palette='magma', ax=axes[0])
axes[0].set_title('Payment Type Usage Count')
axes[0].set_xlabel('Number of Transactions')
axes[0].set_ylabel('Payment Type')

sns.barplot(data=revenue_by_type, x='total_revenue', y='payment_type', palette='rocket', ax=axes[1])
axes[1].set_title('Revenue by Payment Type')
axes[1].set_xlabel('Total Revenue (R$)')
axes[1].set_ylabel('Payment Type')

plt.tight_layout()
plt.show()


## 9. Installment Analysis

In [ ]:
installment_payments = payment[payment['payment_installments'] > 1]
num_installment = installment_payments.shape[0]
installment_rate = (num_installment / len(payment)) * 100

print(f'Orders with installments:           {num_installment}')
print(f'Installment Rate:                   {installment_rate:.2f}%')
print(f'Single Payment Rate:                {100 - installment_rate:.2f}%')

avg_installments = payment['payment_installments'].mean()
print(f'Avg Number of Installments:         {avg_installments:.2f}')

cc_payments = payment[payment['payment_type'] == 'credit_card']
avg_cc_installments = cc_payments['payment_installments'].mean()
print(f'Avg Installments (Credit Card only):{avg_cc_installments:.2f}')

correlation = payment['payment_installments'].corr(payment['payment_value'])
print(f'Correlation installments vs value:  {correlation:.2f}')


In [ ]:
print('Installment distribution (top 15):')
print(payment['payment_installments'].value_counts().sort_index().head(15))


### Installment Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

installment_dist = payment['payment_installments'].value_counts().sort_index().head(15).reset_index()
installment_dist.columns = ['installments', 'count']
sns.barplot(data=installment_dist, x='installments', y='count', palette='viridis', ax=axes[0])
axes[0].set_title('Distribution of Payment Installments')
axes[0].set_xlabel('Number of Installments')
axes[0].set_ylabel('Count')

avg_value_by_installment = payment.groupby('payment_installments')['payment_value'].mean().reset_index().head(15)
sns.lineplot(data=avg_value_by_installment, x='payment_installments', y='payment_value', marker='o', ax=axes[1])
axes[1].set_title('Avg Order Value by Installment Count')
axes[1].set_xlabel('Number of Installments')
axes[1].set_ylabel('Avg Order Value (R$)')

plt.tight_layout()
plt.show()


## 10. Order Value Distribution

In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(payment[payment['payment_value'] <= 1000]['payment_value'], bins=50, kde=True, color='purple')
plt.title('Order Value Distribution (capped at R$1000)')
plt.xlabel('Payment Value (R$)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


## 11. Export for Power BI

In [ ]:
output_path = os.path.join(project_root, 'Data', 'processed')
os.makedirs(output_path, exist_ok=True)

# Row level — for payment type analysis in Power BI
payment.to_csv(os.path.join(output_path, 'payments_cleaned.csv'), index=False)

# Order level — for RFM monetary value
order_level.to_csv(os.path.join(output_path, 'payments_order_level.csv'), index=False)

print('✅ Exported payments_cleaned.csv to Data/processed/')
print('✅ Exported payments_order_level.csv to Data/processed/')
